In [1]:
# ====================================================================
# URDU SIGN LANGUAGE RECOGNITION - SIMPLIFIED VERSION
# ====================================================================

import os
import cv2
import numpy as np
import pandas as pd
import mediapipe as mp
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
import pickle
import json
from datetime import datetime
from tqdm import tqdm

# ====================================================================
# CONFIGURATION
# ====================================================================

# Data paths (from first code)
CSV_FILE = 'dataset/dataset.csv'
VIDEO_DIR = 'dataset/dynamic'
OUTPUT_DIR = 'models_urdu_sl'

# Training settings
SEQUENCE_LENGTH = 30
TEST_SIZE = 0.15
VAL_SIZE = 0.15
RANDOM_STATE = 42

# Model settings
LSTM_UNITS = 64
DROPOUT = 0.6
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 0.001

# Feature dimensions: 2 hands × 21 landmarks × 3 coords × 2 (raw+normalized) + 6 extra = 258
FEATURE_DIM = 258

print("="*70)
print(" URDU SIGN LANGUAGE RECOGNITION")
print("="*70)
print(f" CSV file: {CSV_FILE}")
print(f" Video directory: {VIDEO_DIR}")
print(f" Sequence length: {SEQUENCE_LENGTH} frames")
print(f" Split: Train 70% | Val 15% | Test 15%")
print(f" Feature dimension: {FEATURE_DIM}")
print("="*70)

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ====================================================================
# STEP 1: LOAD DATA FROM CSV
# ====================================================================

print("\n Loading dataset...")

df = pd.read_csv(CSV_FILE, encoding='utf-8')
print(f" Loaded {len(df)} entries from CSV")

# Validate video paths
valid_indices = []
for idx, row in df.iterrows():
    if os.path.exists(row['file']):
        valid_indices.append(idx)

df_valid = df.iloc[valid_indices].reset_index(drop=True)
print(f" Found {len(df_valid)} valid videos")

# Extract paths and labels
video_paths = df_valid['file'].values
labels = df_valid['gesture'].values

# ====================================================================
# STEP 2: SPLIT DATA BEFORE AUGMENTATION
# ====================================================================

print("\n Splitting data (BEFORE augmentation)...")

# First split: separate test set
X_temp, X_test_paths, y_temp, y_test = train_test_split(
    video_paths, labels,
    test_size=TEST_SIZE,
    stratify=labels,
    random_state=RANDOM_STATE
)

# Second split: separate train and validation
X_train_paths, X_val_paths, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=VAL_SIZE / (1 - TEST_SIZE),
    stratify=y_temp,
    random_state=RANDOM_STATE
)

print(f"\n Split complete:")
print(f"   Train: {len(X_train_paths)}")
print(f"   Val:   {len(X_val_paths)}")
print(f"   Test:  {len(X_test_paths)}")

# Verify no overlap
train_set = set(X_train_paths)
val_set = set(X_val_paths)
test_set = set(X_test_paths)

if len(train_set & val_set) == 0 and len(train_set & test_set) == 0:
    print(f"    No data leakage - perfect split!")

# ====================================================================
# AUGMENTATION FUNCTIONS (ONLY MIRROR AND BRIGHTNESS)
# ====================================================================

def adjust_brightness(frame, factor):
    """Adjust brightness of frame"""
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    h, s, v = cv2.split(hsv)
    v = np.clip(v * factor, 0, 255).astype(np.uint8)
    hsv = cv2.merge([h, s, v])
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

def mirror_frame(frame):
    """Horizontal flip of frame"""
    return cv2.flip(frame, 1)

# ====================================================================
# MEDIAPIPE SETUP (TWO HANDS)
# ====================================================================

print("\n Initializing MediaPipe...")

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,  # Two hands
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
    model_complexity=0
)

def extract_landmarks_from_results(results):
    """Extract 258 features from MediaPipe results"""
    if not results.multi_hand_landmarks:
        return None
    
    all_features = []
    
    # Process up to 2 hands
    for i in range(2):
        if i < len(results.multi_hand_landmarks):
            hand_landmarks = results.multi_hand_landmarks[i]
            
            # Raw coordinates (63 features)
            raw_coords = []
            for lm in hand_landmarks.landmark:
                raw_coords.extend([lm.x, lm.y, lm.z])
            all_features.extend(raw_coords)
            
            # Normalized coordinates relative to wrist (63 features)
            wrist = hand_landmarks.landmark[0]
            norm_coords = []
            for lm in hand_landmarks.landmark:
                norm_coords.extend([
                    lm.x - wrist.x,
                    lm.y - wrist.y,
                    lm.z - wrist.z
                ])
            all_features.extend(norm_coords)
        else:
            # Pad with zeros if hand not detected (126 features)
            all_features.extend([0.0] * 126)
    
    # Additional features (6 features)
    num_hands = len(results.multi_hand_landmarks)
    all_features.extend([
        num_hands / 2.0,
        1.0 if num_hands > 0 else 0.0,
        1.0 if num_hands > 1 else 0.0,
        0.5,
        0.5,
        0.0
    ])
    
    return np.array(all_features, dtype=np.float32)

def process_video(video_path, target_frames=SEQUENCE_LENGTH, resize_width=320):
    """Extract features from original video"""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None
    
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if total_frames >= target_frames:
        frame_indices = np.linspace(0, total_frames - 1, target_frames, dtype=int)
    else:
        frame_indices = range(total_frames)
    
    all_landmarks = []
    
    for frame_idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
        ret, frame = cap.read()
        if not ret:
            continue
        
        # Resize frame
        height, width = frame.shape[:2]
        new_height = int(height * (resize_width / width))
        frame_resized = cv2.resize(frame, (resize_width, new_height))
        
        # Process with MediaPipe
        frame_rgb = cv2.cvtColor(frame_resized, cv2.COLOR_BGR2RGB)
        results = hands.process(frame_rgb)
        
        landmarks = extract_landmarks_from_results(results)
        if landmarks is not None:
            all_landmarks.append(landmarks)
    
    cap.release()
    
    if len(all_landmarks) == 0:
        return None
    
    # Pad or trim to target length
    if len(all_landmarks) >= target_frames:
        sequence = all_landmarks[:target_frames]
    else:
        sequence = all_landmarks.copy()
        while len(sequence) < target_frames:
            sequence.append(all_landmarks[-1])
        sequence = sequence[:target_frames]
    
    return np.array(sequence, dtype=np.float32)

def process_video_with_augmentation(video_path, aug_type, brightness_factor=None):
    """Process video with augmentation (mirror or brightness)"""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None
    
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if total_frames >= SEQUENCE_LENGTH:
        frame_indices = np.linspace(0, total_frames - 1, SEQUENCE_LENGTH, dtype=int)
    else:
        frame_indices = range(total_frames)
    
    all_landmarks = []
    
    for frame_idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
        ret, frame = cap.read()
        if not ret:
            continue
        
        # Apply augmentation
        if aug_type == 'mirror':
            frame = mirror_frame(frame)
        elif aug_type == 'brightness' and brightness_factor:
            frame = adjust_brightness(frame, brightness_factor)
        
        # Resize and process
        height, width = frame.shape[:2]
        frame_resized = cv2.resize(frame, (320, int(height * (320 / width))))
        
        frame_rgb = cv2.cvtColor(frame_resized, cv2.COLOR_BGR2RGB)
        results = hands.process(frame_rgb)
        
        landmarks = extract_landmarks_from_results(results)
        if landmarks is not None:
            all_landmarks.append(landmarks)
    
    cap.release()
    
    if len(all_landmarks) == 0:
        return None
    
    # Pad or trim
    if len(all_landmarks) >= SEQUENCE_LENGTH:
        sequence = all_landmarks[:SEQUENCE_LENGTH]
    else:
        sequence = all_landmarks.copy()
        while len(sequence) < SEQUENCE_LENGTH:
            sequence.append(all_landmarks[-1])
        sequence = sequence[:SEQUENCE_LENGTH]
    
    return np.array(sequence, dtype=np.float32)

# ====================================================================
# STEP 3: EXTRACT FEATURES WITH AUGMENTATION
# ====================================================================

print("\n Extracting features...")

def extract_features_with_augmentation(paths, labels, set_name, augment=False):
    """Extract features with augmentation for training set only"""
    features = []
    labels_out = []
    failed = 0
    
    print(f"\n{set_name} set (augment={augment}):")
    
    for path, label in tqdm(zip(paths, labels), total=len(paths), desc="Processing"):
        # Original video
        sequence = process_video(path, SEQUENCE_LENGTH)
        if sequence is not None:
            features.append(sequence)
            labels_out.append(label)
        else:
            failed += 1
        
        # Add augmentations ONLY for training set
        if augment:
            # Mirror augmentation
            aug_seq = process_video_with_augmentation(path, 'mirror')
            if aug_seq is not None:
                features.append(aug_seq)
                labels_out.append(label)
            
            # Darker (brightness 0.8)
            aug_seq = process_video_with_augmentation(path, 'brightness', 0.8)
            if aug_seq is not None:
                features.append(aug_seq)
                labels_out.append(label)
            
            # Brighter (brightness 1.2)
            aug_seq = process_video_with_augmentation(path, 'brightness', 1.2)
            if aug_seq is not None:
                features.append(aug_seq)
                labels_out.append(label)
    
    print(f"    Extracted: {len(features)} (Failed: {failed})")
    if augment:
        print(f"    Augmentation: {len(paths)} → {len(features)} ({len(features)/len(paths):.1f}x)")
    
    return np.array(features, dtype=np.float32), np.array(labels_out)

# Encode labels first
label_encoder = LabelEncoder()
label_encoder.fit(labels)

y_train_encoded = label_encoder.transform(y_train)
y_val_encoded = label_encoder.transform(y_val)
y_test_encoded = label_encoder.transform(y_test)

# Extract features
X_train_features, y_train_final = extract_features_with_augmentation(
    X_train_paths, y_train_encoded, "Train", augment=True  # AUGMENT TRAINING ONLY
)

X_val_features, y_val_final = extract_features_with_augmentation(
    X_val_paths, y_val_encoded, "Validation", augment=False  # NO AUGMENTATION
)

X_test_features, y_test_final = extract_features_with_augmentation(
    X_test_paths, y_test_encoded, "Test", augment=False  # NO AUGMENTATION
)

print(f"\n Feature extraction complete:")
print(f"   Train shape: {X_train_features.shape} (AUGMENTED)")
print(f"   Val shape:   {X_val_features.shape} (ORIGINAL ONLY)")
print(f"   Test shape:  {X_test_features.shape} (ORIGINAL ONLY)")

# Close MediaPipe
hands.close()

# ====================================================================
# STEP 4: BUILD MODEL (REGULAR LSTM)
# ====================================================================

print("\n Building model...")

num_classes = len(label_encoder.classes_)

model = Sequential([
    # Regular LSTM (not bidirectional)
    LSTM(LSTM_UNITS, return_sequences=False, input_shape=(SEQUENCE_LENGTH, FEATURE_DIM)),
    Dropout(DROPOUT),
    
    # Dense layers
    Dense(64, activation='relu'),
    Dropout(DROPOUT),
    
    # Output layer
    Dense(num_classes, activation='softmax')
])

# Compile model
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print(" Model built")
model.summary()

# ====================================================================
# STEP 5: TRAIN MODEL
# ====================================================================

print("\n Training...")

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_name = f'urdu_sl_model_{timestamp}'

callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=0.00001,
        verbose=1
    ),
    ModelCheckpoint(
        filepath=os.path.join(OUTPUT_DIR, f'{model_name}_best.h5'),
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

history = model.fit(
    X_train_features, y_train_final,
    validation_data=(X_val_features, y_val_final),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

# ====================================================================
# STEP 6: EVALUATE MODEL
# ====================================================================

print("\n Evaluating...")

test_loss, test_accuracy = model.evaluate(X_test_features, y_test_final, verbose=0)
best_val_accuracy = max(history.history['val_accuracy'])
best_train_accuracy = max(history.history['accuracy'])

print(f"\n RESULTS:")
print(f"   Train Accuracy:      {best_train_accuracy*100:.2f}%")
print(f"   Validation Accuracy: {best_val_accuracy*100:.2f}%")
print(f"   Test Accuracy:       {test_accuracy*100:.2f}% ← HONEST PERFORMANCE")

gap = best_val_accuracy - test_accuracy
print(f"\n Overfitting Check:")
print(f"   Val-Test Gap: {gap*100:.2f}%")

if gap < 0.05:
    print(f"    EXCELLENT! Model generalizes well")
elif gap < 0.10:
    print(f"    GOOD! Acceptable generalization")
else:
    print(f"    Some overfitting detected")

# Per-class accuracy
print(f"\n Per-gesture accuracy:")
y_pred = model.predict(X_test_features)
y_pred_classes = np.argmax(y_pred, axis=1)

for class_idx, gesture in enumerate(label_encoder.classes_):
    mask = y_test_final == class_idx
    if np.sum(mask) > 0:
        class_acc = np.mean(y_pred_classes[mask] == class_idx)
        print(f"   {gesture}: {class_acc*100:.1f}%")

# ====================================================================
# STEP 7: SAVE MODEL AND METADATA
# ====================================================================

print("\n Saving model...")

# Save model
model_path = os.path.join(OUTPUT_DIR, f'{model_name}.h5')
model.save(model_path)
print(f" Model saved: {model_path}")

# Save label encoder
label_encoder_path = os.path.join(OUTPUT_DIR, f'{model_name}_labels.pkl')
with open(label_encoder_path, 'wb') as f:
    pickle.dump(label_encoder, f)
print(f" Label encoder saved: {label_encoder_path}")

# Save metadata
metadata = {
    'timestamp': timestamp,
    'model_name': model_name,
    'train_samples': len(X_train_features),
    'val_samples': len(X_val_features),
    'test_samples': len(X_test_features),
    'augmentation_factor': len(X_train_features) / len(X_train_paths),
    'num_classes': num_classes,
    'sequence_length': SEQUENCE_LENGTH,
    'feature_dim': FEATURE_DIM,
    'lstm_units': LSTM_UNITS,
    'dropout': DROPOUT,
    'epochs_trained': len(history.history['loss']),
    'best_val_accuracy': float(best_val_accuracy),
    'test_accuracy': float(test_accuracy),
    'val_test_gap': float(gap),
    'class_names': label_encoder.classes_.tolist(),
    'augmentations': ['original', 'mirror', 'brightness_0.8', 'brightness_1.2']
}

metadata_path = os.path.join(OUTPUT_DIR, f'{model_name}_metadata.json')
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=4, ensure_ascii=False)
print(f" Metadata saved: {metadata_path}")

print(f"\n Model saved: {model_path}")
print(f" Labels saved: {label_encoder_path}")

print("\n" + "="*70)
print(" TRAINING COMPLETE WITH PROPER AUGMENTATION!")
print("="*70)
print(f" This model used:")
print(f"    Two-hand extraction (258 features)")
print(f"    Mirror and brightness augmentation only")
print(f"    Regular LSTM architecture")
print(f"    Augmented ONLY training data")
print(f"    Val/Test see only original videos")
print("="*70)

🎯 URDU SIGN LANGUAGE RECOGNITION
📁 CSV file: dataset/dataset.csv
📁 Video directory: dataset/dynamic
📊 Sequence length: 30 frames
🔀 Split: Train 70% | Val 15% | Test 15%
📏 Feature dimension: 258

📂 Loading dataset...
✅ Loaded 670 entries from CSV
✅ Found 670 valid videos

✂️ Splitting data (BEFORE augmentation)...

✅ Split complete:
   Train: 468
   Val:   101
   Test:  101
   ✅ No data leakage - perfect split!

🤖 Initializing MediaPipe...

🎬 Extracting features...

Train set (augment=True):


Processing:   0%|                                                                              | 0/468 [00:00<?, ?it/s]C:\Users\mypc4\AppData\Roaming\Python\Python310\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
Processing: 100%|████████████████████████████████████████████████████████████████████| 468/468 [54:31<00:00,  6.99s/it]


   ✅ Extracted: 1872 (Failed: 0)
   📊 Augmentation: 468 → 1872 (4.0x)

Validation set (augment=False):


Processing: 100%|████████████████████████████████████████████████████████████████████| 101/101 [03:56<00:00,  2.35s/it]


   ✅ Extracted: 101 (Failed: 0)

Test set (augment=False):


...

59/59 ━━━━━━━━━━━━━━━━━━━━ 2s 35ms/step - accuracy: 0.7954 - loss: 0.5964 - val_accuracy: 0.9604 - val_loss: 0.1614 - learning_rate: 0.0010
Epoch 20/50
58/59 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.7878 - loss: 0.5840
Epoch 20: val_accuracy did not improve from 0.96040
59/59 ━━━━━━━━━━━━━━━━━━━━ 2s 33ms/step - accuracy: 0.7959 - loss: 0.5470 - val_accuracy: 0.9505 - val_loss: 0.1821 - learning_rate: 0.0010
Epoch 21/50
59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.8345 - loss: 0.4609
Epoch 21: val_accuracy did not improve from 0.96040
59/59 ━━━━━━━━━━━━━━━━━━━━ 2s 33ms/step - accuracy: 0.8210 - loss: 0.5062 - val_accuracy: 0.9307 - val_loss: 0.2200 - learning_rate: 0.0010
Epoch 22/50
58/59 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.8068 - loss: 0.5077
Epoch 22: val_accuracy did not improve from 0.96040
59/59 ━━━━━━━━━━━━━━━━━━━━ 2s 33ms/step - accuracy: 0.8210 - loss: 0.4905 - val_accuracy: 0.9406 - val_loss: 0.1522 - learning_rate: 0.0010
Epoch 23/50
59/59 ━━━━━━━━

59/59 ━━━━━━━━━━━━━━━━━━━━ 2s 35ms/step - accuracy: 0.8387 - loss: 0.4393 - val_accuracy: 0.9703 - val_loss: 0.1208 - learning_rate: 0.0010
Epoch 25/50
58/59 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.8484 - loss: 0.4269
Epoch 25: val_accuracy did not improve from 0.97030
59/59 ━━━━━━━━━━━━━━━━━━━━ 2s 33ms/step - accuracy: 0.8424 - loss: 0.4434 - val_accuracy: 0.9505 - val_loss: 0.2362 - learning_rate: 0.0010
Epoch 26/50
59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.8534 - loss: 0.4260
Epoch 26: val_accuracy did not improve from 0.97030
59/59 ━━━━━━━━━━━━━━━━━━━━ 2s 33ms/step - accuracy: 0.8387 - loss: 0.4579 - val_accuracy: 0.8911 - val_loss: 0.2030 - learning_rate: 0.0010
Epoch 27/50
58/59 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.8269 - loss: 0.5234
Epoch 27: val_accuracy did not improve from 0.97030
59/59 ━━━━━━━━━━━━━━━━━━━━ 2s 32ms/step - accuracy: 0.8435 - loss: 0.4722 - val_accuracy: 0.9703 - val_loss: 0.1132 - learning_rate: 0.0010
Epoch 28/50
59/59 ━━━━━━━━

59/59 ━━━━━━━━━━━━━━━━━━━━ 2s 33ms/step - accuracy: 0.8643 - loss: 0.3832 - val_accuracy: 0.9901 - val_loss: 0.0635 - learning_rate: 0.0010
Epoch 29/50
58/59 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.8770 - loss: 0.3522
Epoch 29: val_accuracy did not improve from 0.99010
59/59 ━━━━━━━━━━━━━━━━━━━━ 2s 32ms/step - accuracy: 0.8771 - loss: 0.3545 - val_accuracy: 0.9010 - val_loss: 0.2629 - learning_rate: 0.0010
Epoch 30/50
58/59 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.7682 - loss: 0.8805
Epoch 30: val_accuracy did not improve from 0.99010
59/59 ━━━━━━━━━━━━━━━━━━━━ 2s 32ms/step - accuracy: 0.7837 - loss: 0.7869 - val_accuracy: 0.9307 - val_loss: 0.1976 - learning_rate: 0.0010
Epoch 31/50
58/59 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.8440 - loss: 0.4712
Epoch 31: val_accuracy did not improve from 0.99010
59/59 ━━━━━━━━━━━━━━━━━━━━ 2s 33ms/step - accuracy: 0.8595 - loss: 0.4239 - val_accuracy: 0.9802 - val_loss: 0.0878 - learning_rate: 0.0010
Epoch 32/50
59/59 ━━━━━━━━

59/59 ━━━━━━━━━━━━━━━━━━━━ 2s 34ms/step - accuracy: 0.9177 - loss: 0.2485 - val_accuracy: 1.0000 - val_loss: 0.0122 - learning_rate: 5.0000e-04
Epoch 47/50
59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.8988 - loss: 0.3064
Epoch 47: val_accuracy did not improve from 1.00000
59/59 ━━━━━━━━━━━━━━━━━━━━ 2s 34ms/step - accuracy: 0.8862 - loss: 0.3368 - val_accuracy: 1.0000 - val_loss: 0.0102 - learning_rate: 5.0000e-04
Epoch 48/50
59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.9192 - loss: 0.2538
Epoch 48: val_accuracy did not improve from 1.00000
59/59 ━━━━━━━━━━━━━━━━━━━━ 2s 32ms/step - accuracy: 0.9252 - loss: 0.2354 - val_accuracy: 1.0000 - val_loss: 0.0076 - learning_rate: 5.0000e-04
Epoch 49/50
59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.9185 - loss: 0.2529
Epoch 49: val_accuracy did not improve from 1.00000
59/59 ━━━━━━━━━━━━━━━━━━━━ 2s 33ms/step - accuracy: 0.9225 - loss: 0.2364 - val_accuracy: 1.0000 - val_loss: 0.0069 - learning_rate: 5.0000e-04
Epoch 50/5

   آپ بہت خوبصورت ہیں۔: 100.0%
   آپ کا نام کیا ہے؟: 100.0%
   آہِستہ: 100.0%
   ادھر آو!: 100.0%
   السلام وعلیکم: 100.0%
   آپ کہاں رہتے ہیں؟: 100.0%
   آپ کیسے ہیں؟: 100.0%
   ایمبولینس کو کال کریں: 100.0%
   بھائی: 100.0%
   حالات کیسے ہیں؟: 100.0%
   خوش رہو: 100.0%
   شکریہ: 100.0%
   فون کہاں ہے؟: 100.0%
   مبارک ہو: 100.0%
   میں اشاروں کی زبان سیکھ رہا ہوں: 100.0%
   میں واش روم جا سکتا ہوں؟: 100.0%
   میں ٹھیک ہوں: 100.0%
   نہیں: 100.0%
   چھونا مت: 100.0%
   کمپیوٹر: 100.0%
   کیا آپ تیار ہیں؟: 100.0%
   کیا آپ پاس یا ناکام ہوئے ہیں؟: 100.0%
   کیا تم بہرے ہو؟: 100.0%
   کیا میں اپ کی مدد کر سکتا ہوں؟: 100.0%
   گھبرائیں نہیں: 100.0%

💾 Saving model...
✅ Model saved: models_urdu_sl\urdu_sl_model_20251026_001348.h5
✅ Label encoder saved: models_urdu_sl\urdu_sl_model_20251026_001348_labels.pkl
✅ Metadata saved: models_urdu_sl\urdu_sl_model_20251026_001348_metadata.json

✅ Model saved: models_urdu_sl\urdu_sl_model_20251026_001348.h5
✅ Labels saved: models_urdu_sl\urdu_sl_mod